<a href="https://colab.research.google.com/github/csu-techhub/QOS-Labs/blob/main/Lab6.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 6 — H₂ Bond-Length Curve, Noise, and Optional Hardware
**Quantum Optimization and Simulation — VQE Laboratory Series**

Connect VQE's electronic inner loop to geometry optimization.

In Lab 5 we ran a complete VQE for H₂ at **one** bond length, $R = 0.735$ Å, and found the ground-state energy $-1.137$ Ha. But why 0.735 Å? Would the molecule be more stable if the two atoms were closer together or farther apart? In this lab we answer that question by repeating VQE at many bond lengths and drawing the **potential-energy curve**. Then we see what happens when the same calculation runs on a noisy (simulated) quantum computer.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`. In this notebook every bitstring we *discuss* is written in the logical order `q0 q1` (left to right), as in Labs 4 and 5.

## Learning objectives
- Distinguish the **inner loop** (electronic structure: adjust $\theta$ at a fixed bond length $R$) from the **outer loop** (geometry optimization: adjust $R$).
- Construct an H₂ potential-energy curve from Hamiltonians supplied at several bond lengths.
- Compare three estimates of the same curve: **exact**, **noiseless shot-based VQE**, and **noisy VQE** on a simulated IBM device (FakeSherbrooke).

## The big picture: two loops
This is the overall VQE process from Lesson 5 of the lecture:

| | **Inner loop** — electronic structure | **Outer loop** — geometry optimization |
|---|---|---|
| What changes | the control knob $\theta$ (the configuration mix) | the bond length $R$ |
| What stays fixed | the Hamiltonian (one fixed geometry) | — the Hamiltonian **changes** with $R$ |
| Who adjusts it | COBYLA, exactly as in Lab 5 | here: a simple scan over a list of $R$ values |
| Result | the lowest energy at that $R$ | the $R$ with the lowest energy overall (the equilibrium bond length) |

Lab 5 was one inner loop. Lab 6 wraps an outer loop around it.

In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install "pylatexenc" "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8" "qiskit-ibm-runtime~=0.46" "python-dateutil<3"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

SEED = 123
SHOTS = 4096
CHEM_ACC = 1.6e-3            # chemical accuracy: 1.6 mHa

# Helper functions
def run_counts(qc, shots=SHOTS, backend=None):
    sim = AerSimulator() if backend is None else backend      # ideal simulator unless a backend is given
    tqc = transpile(qc, sim, optimization_level=1)              # translate into the backend's native gates
    result = sim.run(tqc, shots=shots, seed_simulator=SEED).result()
    return result.get_counts()

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]

In [ ]:
def ansatz(theta):
    qc = QuantumCircuit(2)

    # Prepare |01> (q0-first): the Hartree-Fock (bonding) reference
    qc.x(1)

    # ------------------------------------------------
    # RYX(theta) = exp[-i theta/2 (Y0 X1)]
    # ------------------------------------------------
    qc.sdg(0)
    qc.h(0)
    qc.h(1)

    # ZZ rotation
    qc.cx(0, 1)
    qc.rz(theta, 1)
    qc.cx(0, 1)

    # Undo basis change
    qc.h(0)
    qc.s(0)
    qc.h(1)

    # ------------------------------------------------
    # RXY(-theta) = exp[+i theta/2 (X0 Y1)]
    # ------------------------------------------------

    qc.h(0)
    qc.sdg(1)
    qc.h(1)

    # ZZ rotation with angle -theta
    qc.cx(0, 1)
    qc.rz(-theta, 1)
    qc.cx(0, 1)

    # Undo basis change
    qc.h(0)
    qc.h(1)
    qc.s(1)
    return qc

### Measuring the energy (the same recipe as Lab 5, Part A)
For a trial state, the energy expectation value is

$$E = c_0 + c_1\langle Z_0\rangle + c_2\langle Z_1\rangle + c_3\langle Z_0Z_1\rangle + c_4\big(\langle X_0X_1\rangle + \langle Y_0Y_1\rangle\big) + E_{\text{nuc}} .$$


In [ ]:
def parity_expectation(counts, qubits):
    total = sum(counts.values())
    result = 0.0
    for bits, count in counts.items():
        value = 1
        for q in qubits:
            bit = int(bits[-1-q])
            value *= 1 if bit == 0 else -1
        result += value * count/total
    return result

theta_sym = Parameter("θ")

def measurement_circuits(backend):
    "The Z-, X- and Y-basis circuits, transpiled once for this backend (theta left symbolic)."
    circuits = {}
    for basis in ["Z", "X", "Y"]:
        qc = ansatz(theta_sym)
        if basis == "X":                 # HZH = X
            qc.h([0, 1])
        elif basis == "Y":               # S H Z H S† = Y
            qc.sdg([0, 1])
            qc.h([0, 1])
        qc.measure_all()
        circuits[basis] = transpile(qc, backend, optimization_level=1, seed_transpiler=SEED)
    return circuits

def sampled_energy(theta_array, coeffs, backend, circuits):
    "Energy expectation value (total, Hartree) estimated from SHOTS measurements per circuit."
    theta = float(np.atleast_1d(theta_array)[0])
    c0, c1, c2, c3, c4, e_nuc = coeffs
    bases = ["Z", "X", "Y"]
    bound = [circuits[b].assign_parameters({theta_sym: theta}) for b in bases]
    result = backend.run(bound, shots=SHOTS, seed_simulator=SEED).result()   # all three circuits in one call
    counts = {b: result.get_counts(i) for i, b in enumerate(bases)}
    z0 = parity_expectation(counts["Z"], [0])
    z1 = parity_expectation(counts["Z"], [1])
    zz = parity_expectation(counts["Z"], [0, 1])
    xx = parity_expectation(counts["X"], [0, 1])
    yy = parity_expectation(counts["Y"], [0, 1])
    return c0 + c1*z0 + c2*z1 + c3*zz + c4*(xx + yy) + e_nuc

## Coefficient sets: one Hamiltonian per bond length
As the lecture put it: in the outer loop we *adjust $R$, and the Hamiltonian changes accordingly*. When the atoms move, the electrons feel the protons differently, so every coefficient changes.

The table below gives, for each bond length $R$, the five coefficients $c_0 \dots c_4$ and the nuclear repulsion $E_{\text{nuc}}$. They were generated with PySCF + Qiskit Nature, exactly the procedure in the optional part of Lab 5, using the same qubit convention as the lecture. The $R = 0.735$ Å row is the Hamiltonian from Lab 5 and the lecture slides.

In [ ]:
# R in Angstrom: (c0, c1, c2, c3, c4, E_nuc), all in Hartree
h2_data = {
    0.500: (-0.9477078770, -0.5830796255, +0.5830796255, -0.0125164316, +0.0844351138, 1.0583544218),
    0.600: (-1.0071270770, -0.4940137866, +0.4940137866, -0.0120643897, +0.0868653219, 0.8819620182),
    0.700: (-1.0439125218, -0.4204556798, +0.4204556798, -0.0115074022, +0.0895002880, 0.7559674442),
    0.735: (-1.0523732458, -0.3979374248, +0.3979374248, -0.0112801043, +0.0904655999, 0.7199689944),
    0.800: (-1.0632127971, -0.3599594245, +0.3599594245, -0.0108097350, +0.0923133918, 0.6614715137),
    0.900: (-1.0702832691, -0.3097872795, +0.3097872795, -0.0099691083, +0.0952858469, 0.5879746788),
    1.000: (-1.0692434904, -0.2675286499, +0.2675286499, -0.0090149301, +0.0983952917, 0.5291772109),
    1.200: (-1.0526707246, -0.2001895768, +0.2001895768, -0.0069621624, +0.1048957343, 0.4409810091),
    1.400: (-1.0253590356, -0.1495679388, +0.1495679388, -0.0050305364, +0.1115110445, 0.3779837221),
    1.700: (-0.9767372365, -0.0958445867, +0.0958445867, -0.0028080706, +0.1210364193, 0.3112807123),
    2.000: (-0.9285563459, -0.0606280089, +0.0606280089, -0.0014311039, +0.1295692374, 0.2645886055),
}
R_values = list(h2_data.keys())

# How the Hamiltonian changes with R
labels = ["c1 (Z0)", "c2 (Z1)", "c4 (hopping)", "E_nuc (proton repulsion)"]
cols = [1, 2, 4, 5]
plt.figure(figsize=(8, 4))
for lab, k in zip(labels, cols):
    plt.plot(R_values, [h2_data[R][k] for R in R_values], "o-", label=lab)
plt.axhline(0, color="gray", lw=0.8)
plt.xlabel("Bond distance R (Angstrom)"); plt.ylabel("Coefficient (Hartree)")
plt.title("The Hamiltonian changes with the bond length"); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.show()

Two things to notice in this chart:
- **$E_{\text{nuc}}$ grows quickly as the atoms get closer.** The protons repel each other strongly at short distance. This is what stops the bond from collapsing.
- **$c_1$ and $c_2$ shrink as the bond stretches, while the hopping coefficient $c_4$ grows.** The bonding and antibonding orbitals come closer in energy, so mixing in the antibonding configuration becomes more important at long distances.

## Part A — Noiseless VQE at every bond length (inner loop + outer loop)
Now the real VQE. For **each** $R$ (outer loop), we run the **same** COBYLA optimization as Lab 5 (inner loop), starting from the Hartree–Fock state $\theta = 0$. The energies are estimated from 4096 shots per circuit on the **ideal** simulator: no hardware noise, only the statistical shot noise we met in Lab 5.

This takes under a minute.

In [ ]:
ideal_backend = AerSimulator()
ideal_circuits = measurement_circuits(ideal_backend)

ideal_curve, ideal_theta = {}, {}
print(f"{'R (Å)':>7}{'VQE (Ha)':>12}{'theta':>9}")
for R, coeffs in h2_data.items():                       # outer loop: bond length
    result = minimize(                                   # inner loop: control knob theta
        sampled_energy, x0=[0.0], args=(coeffs, ideal_backend, ideal_circuits),
        method="COBYLA", options={"maxiter": 45, "rhobeg": 0.4},
    )
    ideal_curve[R], ideal_theta[R] = result.fun, result.x[0]
    print(f"{R:7.3f}{ideal_curve[R]:12.6f}{ideal_theta[R]:9.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]
ax.plot(R_values, [ideal_curve[R] for R in R_values], "s--", label="Noiseless shot-based VQE")
best_vqe_R = min(ideal_curve, key=ideal_curve.get)
ax.plot(best_vqe_R, ideal_curve[best_vqe_R], "r*", ms=15, label=f"VQE minimum: R = {best_vqe_R} Å")
ax.set_xlabel("Bond distance R (Angstrom)"); ax.set_ylabel("Ground-state energy (Hartree)")
ax.set_title("H2 potential-energy curve: noiseless VQE"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(R_values, [np.sin(ideal_theta[R])**2 for R in R_values], "o-", color="tab:orange")
ax.set_xlabel("Bond distance R (Angstrom)"); ax.set_ylabel("P(10): antibonding share")
ax.set_title("How much antibonding configuration VQE mixes in"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

This is the famous shape of a chemical bond:
- **Too close (left):** the proton repulsion dominates and the energy shoots up.
- **Too far (right):** the atoms barely interact and the energy levels off toward two separate hydrogen atoms.
- **In between:** a minimum, the **equilibrium bond length**, close to 0.735 Å. That is why Lab 5 used 0.735 Å.
- **Right panel:** near equilibrium, VQE mixes in only about 2–3% of the antibonding configuration (Lab 5 found about 2% at 0.735 Å). As the bond stretches, that share grows a lot, to roughly 30% at 2.0 Å. A stretched bond is much less like the simple Hartree–Fock picture, and the control knob $\theta$ has to work harder.

## Part B — The same calculation on a noisy (simulated) IBM device
So far the simulator was perfect. Real quantum computers are not: every gate has a small chance of error, and measurements sometimes read 0 as 1 (or the reverse).

`FakeSherbrooke` is a snapshot of a real IBM quantum computer, *ibm_sherbrooke*: its qubit layout, its native gates, and its measured error rates. `AerSimulator.from_backend(...)` builds a simulator that behaves like that device, including its noise, without waiting in a queue. Nothing else changes: same Hamiltonians, same ansatz, same COBYLA settings.

First, let's see how our small circuit looks once it is translated for the device.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

fake_sherbrooke = FakeSherbrooke()
sherbrooke_simulator = AerSimulator.from_backend(fake_sherbrooke)   # simulator with Sherbrooke's noise
sherbrooke_circuits = measurement_circuits(sherbrooke_simulator)

z_circ = sherbrooke_circuits["Z"]
print("Our circuit as written :", dict(ansatz(theta_sym).count_ops()))
print("After transpiling      :", dict(z_circ.count_ops()))
print("Two-qubit gates (ECR)  :", z_circ.count_ops().get("ecr", 0))

The device does not have CNOT as a native gate. Each CNOT becomes an **ECR** gate plus single-qubit rotations. Two-qubit gates are the noisiest operations on today's hardware, so these are where most errors come from.

Now run the outer and inner loops again, this time on the noisy simulator. Simulating a device's noise is much slower than ideal simulation, so this takes about 3–5 minutes. A good moment for a coffee.

In [ ]:
sherbrooke_curve, sherbrooke_theta = {}, {}
print(f"{'R (Å)':>7}{'noisy VQE (Ha)':>16}{'exact (Ha)':>12}{'error (mHa)':>13}{'theta':>9}")
for R, coeffs in h2_data.items():                       # outer loop: bond length
    result = minimize(                                   # inner loop: control knob theta
        sampled_energy, x0=[0.0], args=(coeffs, sherbrooke_simulator, sherbrooke_circuits),
        method="COBYLA", options={"maxiter": 45, "rhobeg": 0.4},
    )
    sherbrooke_curve[R], sherbrooke_theta[R] = result.fun, result.x[0]
    print(f"{R:7.3f}{sherbrooke_curve[R]:16.6f}{ideal_curve[R]:12.6f}{1000*(sherbrooke_curve[R]-ideal_curve[R]):13.2f}{sherbrooke_theta[R]:9.3f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(R_values, [ideal_curve[R] for R in R_values], "s--", label="Noiseless shot-based VQE")
plt.plot(R_values, [sherbrooke_curve[R] for R in R_values], "o-", label="FakeSherbrooke VQE (noisy)")
best_noisy_R = min(sherbrooke_curve, key=sherbrooke_curve.get)
plt.plot(best_noisy_R, sherbrooke_curve[best_noisy_R], "r*", ms=15, label=f"noisy minimum: R = {best_noisy_R} Å")
plt.xlabel("Bond distance R (Angstrom)"); plt.ylabel("Ground-state energy (Hartree)")
plt.title("H2 potential-energy curve: noiseless vs noisy")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**What to notice**
- The noisy curve sits the ideal, noiseless curve at every bond length. Gate and readout errors push the measured state away from the ideal one, and any wrong state has a higher energy (the variational principle).
- Even so, the **shape** mostly survives: the noisy curve still has a clear minimum, close to the right bond length (it may shift by one grid point, for example to 0.80 Å). A noisy VQE can roughly answer "where is the bond?" long before it can answer "what is the exact energy?".
- Closing this gap is the job of **error mitigation** (for example, correcting readout errors or extrapolating to zero noise), and in the long run, of better hardware.

## Reflection
Explain the difference between:
- the inner VQE loop at fixed $R$, and
- the outer geometry loop over $R$.

The inner loop changes circuit parameters to find the lowest electronic energy for one fixed molecular geometry. The outer loop changes the molecular geometry itself and repeats the electronic calculation.

**Two more things to think about** (from the lecture)
1. *Is $R$ the only geometry parameter?* For H₂, yes: a two-atom molecule has only one distance. For H₂O the outer loop must adjust two O–H distances and the H–O–H angle, and for larger molecules many atomic coordinates at once.
2. *VQE is expensive.* Count the measurements in this lab: about 20 COBYLA evaluations × 3 circuits × 4096 shots ≈ 250,000 shots for **one** bond length, and 11 bond lengths for the curve, run twice (noiseless and noisy). That is about 5 million shots for the smallest molecule there is. The cost is worth it only because, for large molecules, VQE avoids writing down the enormous matrix that exact diagonalization would need.